In [4]:
from dataloader import Synthetic_linear_data, Real_data, Real_data_MCAR
from experiments import Missing_experiments_MCAR, Missing_experiments
import numpy as np
import wandb
import time
import argparse
from utils import str2bool, positive_int, get_run_name
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

config = {"train_size": 500,
          "cal_size": 100,
          "test_size_iid": 2000,
          "test_size_per_pattern": 100,
          "test_size_per_num_pattern": 100,
          "imputation": 'iterative_ridge',
          "n_rep": 500,
          "method": 'CQR_MDA',
          "basemodel": 'RF',
          "dim": 10,
          "regression": 'Linear',
          "x_mean": 1,
          "phi": 0.8,
          "prob_missing":0.5,
          "var_missing": [1,1,1,1,1,1,1,1,1,1],
          "mechanism": 'MCAR',
          "cores": 8,
          "other_model_parameters": {'n_estimators':100,'min_samples_leaf':3,'max_features':'sqrt'},
          "exact": True,
          "alpha": 0.1,
          "dataset_name": 'synthetic',
          'accept_reject': False,
          'reweight': False,
          'RN_model': 'histgbdt',
          'pre_imputation':'MICE',
          'ocacle_pre_impute':False
          }

run_name = ''

if config['method']=='CQR_MDA':
    if config['accept_reject']:
        run_name += 'ARC-CP_'
    elif config['reweight']:
        run_name += 'Weighted-CP_'
    else:
        if config['exact']:
            run_name += 'Exact_'
        else:
            run_name += 'Nested_'
else:
    run_name += config['method']

run_name += 'nocorrection'
pre_impute = True
data = Synthetic_linear_data(n_rep=config['n_rep'],
                             train_size=config['train_size'],
                             cal_size=config['cal_size'],
                             test_size_iid=config['test_size_iid'],
                             test_size_per_pattern=config['test_size_per_pattern'],
                             mechanism=config['mechanism'],
                             missing_prob=config['prob_missing'],
                             var_missing=np.array(config['var_missing']), var_dependent=[],
                             rho=config['phi'],)



In [5]:
_ = data.generate_multiple_data()

synthetic_linear_500_500_100_2000_100_MCAR_50.0_1111111111__False_dim10_rho0.8


In [6]:
run_name += data.unique_name
print(run_name)
print(config['mechanism'])

exp = Missing_experiments_MCAR(data, pre_impute, config['pre_imputation'], config['imputation'],
                               config['accept_reject'], config['reweight'], config['RN_model'], False,
                               False, 0,
                               config['basemodel'], config['alpha'], config['cores'], config['exact'],
                               {'n_estimators': 100, 'min_samples_leaf': 3, 'max_features': 'sqrt'},
                               ocacle_pre_impute=config['ocacle_pre_impute'])

exp.run_experiments()

Exact_nocorrectionsynthetic_linear_500_500_100_2000_100_MCAR_50.0_1111111111__dim10_rho0.8
MCAR
0 rep over 500
1 rep over 500
2 rep over 500
3 rep over 500
4 rep over 500
5 rep over 500
6 rep over 500
7 rep over 500
8 rep over 500
9 rep over 500
10 rep over 500
11 rep over 500
12 rep over 500
13 rep over 500
14 rep over 500
15 rep over 500
16 rep over 500
17 rep over 500
18 rep over 500
19 rep over 500
20 rep over 500
21 rep over 500
22 rep over 500
23 rep over 500
24 rep over 500
25 rep over 500
26 rep over 500
27 rep over 500
28 rep over 500
29 rep over 500
30 rep over 500
31 rep over 500
32 rep over 500
33 rep over 500
34 rep over 500
35 rep over 500
36 rep over 500
37 rep over 500
38 rep over 500
39 rep over 500
40 rep over 500
41 rep over 500
42 rep over 500
43 rep over 500
44 rep over 500
45 rep over 500
46 rep over 500
47 rep over 500
48 rep over 500
49 rep over 500
50 rep over 500
51 rep over 500
52 rep over 500
53 rep over 500
54 rep over 500
55 rep over 500
56 rep over 500
57

{'iid': {'Y_inf': array([[ 3.18237493e+00,  3.44335510e+00,  1.87729300e+00, -4.33389083e+00,  3.75106836e+00, -8.34770839e+00, -1.89652859e+00, -3.37654509e+00,  6.58412364e-01,  3.19040928e-01,  7.17708664e+00,
          -1.23628148e+00, -3.60279773e+00,  3.40439160e+00, -4.83139816e+00, ...,  9.82993122e-01,  4.50179638e+00,  1.10223811e+00,  8.72315027e+00,  2.94634514e+00, -5.55736879e+00,
          -7.37793969e+00,  9.56056970e+00, -8.96055319e+00, -9.02856080e+00, -3.41944548e+00,  6.72298779e+00,  1.13183963e+00, -7.80207134e+00,  1.05588021e+01],
         [-1.08190618e+01, -3.71624814e+00,  1.22274965e+00,  1.46604425e+00,  2.84722251e+00,  2.81756030e+00,  4.22469156e+00,  5.93094823e-01,  9.12983403e+00,  7.60847301e+00,  3.56211009e+00,
           7.10029704e-01,  4.73414739e+00,  9.47441193e+00,  1.18929253e+00, ...,  1.94211914e+00,  3.68683853e+00, -3.95370564e-01, -9.20229953e-01,  1.57099248e+00,  5.89618871e+00,
           6.46339051e+00, -6.90062544e+00,  4.55044415e

In [ ]:
results = exp.calculate_results_pattern()

In [16]:
print("The worst case missingness pattern coverage: {:.2f}".format(results[1]['mask_cov_worst']))

The worst case missingness pattern coverage: 0.89


In [15]:
print(results[1]['mask_cov_worst'] - 1.96*results[1]['mask_cov_worst_std']/np.sqrt(500), 
      results[1]['mask_cov_worst'] + 1.96*results[1]['mask_cov_worst_std']/np.sqrt(500))

0.8848941239975637 0.8926258760024363


We see that under the same dataset, if we just pre-impute all the values, and don't do a correction, there will be a miscoverage. However, as we have shown in the paper, if we use our ARC-CP or Weighted CP, we will have the desired coverage for all missingness patterns.